In [ ]:
!pip3 install great_expectations

In [2]:
import great_expectations as gx
import pandas as pd

In [3]:
df = pd.read_csv(
    "https://raw.githubusercontent.com/great-expectations/gx_tutorials/main/data/yellow_tripdata_sample_2019-01.csv"
)

# Using Great Expectations

In [4]:
## Example Setup

context = gx.get_context()
data_source = context.data_sources.add_pandas("pandas")
data_asset = data_source.add_dataframe_asset(name="pd dataframe asset")

batch_definition = data_asset.add_batch_definition_whole_dataframe("batch definition")
batch = batch_definition.get_batch(batch_parameters={"dataframe": df})

expectation = gx.expectations.ExpectColumnValuesToBeBetween(
    column="passenger_count", min_value=1, max_value=6
)

validation_result = batch.validate(expectation)

INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmp7lomguum' for ephemeral docs site


Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

#**To-do:**

1.   Fetch any dataset from online source. I'll recommend using reddit API used in the assignment #1.
2.   Run 5 expectations of your choice to validate the dataset. They should cover row, column, multi-column, table, distribution operation.
3.   Submit the notebook file on LMS before Monday 11:55 PM.
4.   Mention your group number in the name of the file.



In [17]:
# Fetching data via reddit api on the topic: German Election Results
import requests

# Reddit API Credentials
client_id = "iySFt69f8BUK6aAyJj0qhQ"
client_secret = "Fi2DdcQE9aw5Ocs51xE1mGwAKgcLgw"
user_agent = "ashar by ashar08"
url = "https://oauth.reddit.com/search"
headers = {
    "User-Agent": user_agent,
}

# Authenticate with the Reddit API
auth = requests.auth.HTTPBasicAuth(client_id, client_secret)
data = {"grant_type": "password", "username": "ashar**", "password": "********"}
response = requests.post("https://www.reddit.com/api/v1/access_token", auth=auth, data=data, headers=headers)

# Get the access token from the response
token = response.json()["access_token"]

# Update the headers to include the access token
headers["Authorization"] = f"bearer {token}"

# Parameters for the search query
params = {
    "q": "german election results",
    "sort": "relevance",
    "limit": 100
}

# Send the request to the Reddit API
response = requests.get(url, headers=headers, params=params)
if response.status_code == 200:
    data = response.json()
    posts = data["data"]["children"]
    post_data = []
    for post in posts:
        post_data.append({
            "title": post["data"]["title"],
            "score": post["data"]["score"],
            "num_comments": post["data"]["num_comments"],
            "url": post["data"]["url"],
        })
    df = pd.DataFrame(post_data)
    print(df)
    df.to_csv("reddit_data.csv", index=False)
else:
    print(f"Error fetching data from Reddit API: {response.status_code}")

                                                title  score  num_comments  \
0                        2025 German federal election    658          3572   
1                  German election results megathread     72           302   
2   Final German election results, SPD wins for th...  17289          2981   
3          German EU-Election results compared by age   2357           699   
4   German election live updates: Voting begins af...    234           369   
..                                                ...    ...           ...   
95  Election results of the 1912 Imperial Election...    283           165   
96  AfD remain on course for record result in YouG...      5             7   
97         I’m tired of the German election hysteria.     19             6   
98  German election: Merz's conservatives celebrat...      5             3   
99  2025 German federal election, but the far-righ...    206            34   

                                                  url  
0   htt

In [20]:
df.head()

,title,score,num_comments,url
0,2025 German federal election,658,3572,https://www.reddit.com/r/europe/comments/1iw72...
1,German election results megathread,72,302,https://www.reuters.com/graphics/GERMANY-ELECT...
2,"Final German election results, SPD wins for th...",17289,2981,https://i.redd.it/i93z82ponyp71.jpg
3,German EU-Election results compared by age,2357,699,https://i.redd.it/8fh4alyttl031.jpg
4,German election live updates: Voting begins af...,234,369,https://www.independent.co.uk/news/world/europ...


In [30]:
# Using Great Expectations
context = gx.get_context()
data_source = context.data_sources.add_pandas("pandas")
data_asset = data_source.add_dataframe_asset(name="reddit_data")
batch_definition = data_asset.add_batch_definition_whole_dataframe("batch definition")
batch = batch_definition.get_batch(batch_parameters={"dataframe": df})

# My Expectations
expectation1 = gx.expectations.ExpectColumnValuesToNotBeNull(column="title")
expectation2 = gx.expectations.ExpectColumnValuesToBeBetween(column="score", min_value=0)
expectation3 = gx.expectations.ExpectColumnValuesToBeBetween(column="num_comments", min_value=0)
expectation4 = gx.expectations.ExpectTableRowCountToEqual(value=len(df))
expectation5 = gx.expectations.ExpectColumnValuesToMatchRegex(
    column="title",
    regex=r".*\b(AfD|SPD|CDU|GRÜNE)\b.*",  # Matches any of AfD, SPD, GRÜNE or CDU as whole words
    mostly=0.1
)

# Create an expectation suite
from great_expectations.core.expectation_suite import ExpectationSuite
expectation_suite = ExpectationSuite(name="reddit_expectation_suite")
for exp in [expectation1, expectation2, expectation3, expectation4, expectation5]:
    expectation_suite.add_expectation(exp)

# Validate the batch with the expectation suite
validation_result = batch.validate(expectation_suite)

INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmp__7j4czn' for ephemeral docs site


Calculating Metrics:   0%|          | 0/27 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/great_expectations/expectations/metrics/column_map_metrics/column_values_match_regex.py:25: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return column.astype(str).str.contains(regex)


In [32]:
print(validation_result)

{
  "success": true,
  "results": [
    {
      "success": true,
      "expectation_config": {
        "type": "expect_column_values_to_not_be_null",
        "kwargs": {
          "batch_id": "pandas-reddit_data",
          "column": "title"
        },
        "meta": {}
      },
      "result": {
        "element_count": 100,
        "unexpected_count": 0,
        "unexpected_percent": 0.0,
        "partial_unexpected_list": [],
        "partial_unexpected_counts": [],
        "partial_unexpected_index_list": []
      },
      "meta": {},
      "exception_info": {
        "raised_exception": false,
        "exception_traceback": null,
        "exception_message": null
      }
    },
    {
      "success": true,
      "expectation_config": {
        "type": "expect_column_values_to_match_regex",
        "kwargs": {
          "batch_id": "pandas-reddit_data",
          "column": "title",
          "mostly": 0.1,
          "regex": ".*\\b(AfD|SPD|CDU|GR\u00dcNE)\\b.*"
        },
        

In [34]:
from great_expectations.render import DefaultJinjaPageView
from great_expectations.render.renderer import ValidationResultsPageRenderer

# Render results in HTML format
validation_result.meta["run_id"] = "my_run_id"
renderer = ValidationResultsPageRenderer()
rendered_content = renderer.render(validation_result)
html = DefaultJinjaPageView().render(rendered_content)

# Display in Colab
from IPython.core.display import display, HTML
display(HTML(html))


,
Evaluated Expectations,5
Successful Expectations,5
Unsuccessful Expectations,0
Success Percent,100%
,
Great Expectations Version,1.3.7
Run Name,my_run_id
Run Time,__none__
,
ge_load_time,20250224T170904.665423Z
